# Notebook 02_06 — Feature Selection: Consensus

Runs **Consensus** across K = {4, 8, 16} × 6 classifiers × 5 seeds.
Uses the pre-computed ranking from `02_00_fs_rankings.ipynb` (`models/fs_rankings.pkl`).

**Results saved incrementally to** `results/fs_consensus_raw.csv` — if this notebook crashes mid-run, just re-run it: completed (K, seed, classifier) combinations are skipped automatically.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
FEATURE_NAMES = d['feature_names']

with open(f'{MODELS_DIR}fs_rankings.pkl', 'rb') as f:
    RANKINGS = pickle.load(f)

METHOD_NAME = 'Consensus'
ranked_indices = RANKINGS[METHOD_NAME]

print(f'Method: {METHOD_NAME}')
print('Top-16 features:', [FEATURE_NAMES[i] for i in ranked_indices[:16]])

Method: Consensus
Top-16 features: ['arp.opcode', 'tcp.dstport', 'ip.version', 'frame.len', 'wlan.fc.subtype', 'radiotap.channel.freq', 'tcp.srcport', 'frame.encap_type', 'udp.dstport', 'radiotap.length', 'wlan_radio.phy', 'tcp.time_relative', 'ip.proto', 'tcp.seq', 'ip.ttl', 'udp.srcport']


## Transform function (slice to top-K features by this method's ranking)

In [3]:
def transform_fn(K):
    selected = ranked_indices[:K]
    return X_train[:, selected], X_val[:, selected], X_test[:, selected], K

## Run experiment grid (resumable)

In [4]:
RESULTS_CSV = f'{RESULTS_DIR}fs_consensus_raw.csv'

run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[Consensus] K=4 seed=42 DT     F1=0.7890  train=0.1s  inf=0.0001ms/sample  (1/90)
[Consensus] K=4 seed=42 RF     F1=0.7609  train=1.2s  inf=0.0028ms/sample  (2/90)
[Consensus] K=4 seed=42 XGB    F1=0.7439  train=1.9s  inf=0.0012ms/sample  (3/90)
[Consensus] K=4 seed=42 LGBM   F1=0.7869  train=1.2s  inf=0.0033ms/sample  (4/90)
[Consensus] K=4 seed=42 KNN    F1=0.5647  train=0.1s  inf=0.0819ms/sample  (5/90)
[Consensus] K=4 seed=42 MLP    F1=0.6332  train=51.3s  inf=0.0018ms/sample  (6/90)
[Consensus] K=4 seed=52 DT     F1=0.7906  train=0.1s  inf=0.0001ms/sample  (7/90)
[Consensus] K=4 seed=52 RF     F1=0.7604  train=1.3s  inf=0.0028ms/sample  (8/90)
[Consensus] K=4 seed=52 XGB    F1=0.7679  train=1.9s  inf=0.0012ms/sample  (9/90)
[Consensus] K=4 seed=52 LGBM   F1=0.7832  train=1.2s  inf=0.0033ms/sample  (10/90)
[Consensus] K=4 seed=52 KNN    F1=0.5723  train=0.1s  inf=0.0654ms/sample  (11/90)
[Consensus] K=4 seed=52 MLP    F1=0.6146  train=38.9s  inf=0.0018ms/sample  (12/90)
[Consensus]

## Quick summary

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
summary = res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std'])
print(summary.to_string())

                     f1           train_time_s            inference_ms          
                   mean       std         mean        std         mean       std
K  classifier                                                                   
4  DT          0.789224  0.000880     0.059437   0.002945     0.000079  0.000002
   KNN         0.564010  0.009801     0.084338   0.001403     0.082044  0.016991
   LGBM        0.643259  0.316006     1.165794   0.058802     0.003017  0.000751
   MLP         0.648560  0.036976    45.800287   7.961321     0.001812  0.000096
   RF          0.773162  0.016590     1.248809   0.024817     0.002725  0.000050
   XGB         0.734291  0.023504     1.807491   0.080929     0.001193  0.000039
8  DT          0.811172  0.000639     0.051975   0.002435     0.000062  0.000002
   KNN         0.578787  0.006792     0.152117   0.004254     0.036027  0.004408
   LGBM        0.810980  0.000579     1.249106   0.078651     0.003022  0.000068
   MLP         0.788085  0.0